# ADMET-Engine v3.0 (*CardioPhys-3D*) — 1-Click All-in-One Colab Pro Master Pipeline

Click **`Runtime -> Run all`** to execute the complete **CardioPhys-3D** research pipeline end-to-end in a single Colab Pro GPU session (T4 / L4 / A100):

| Stage | Work Package | Description |
|---|---|---|
| **0** | Setup & Optional Drive Sync | Clones repo, installs `rdkit` & `torch-geometric`, verifies CUDA GPU, optional non-fatal Drive backup |
| **1** | **WP1** Multi-Channel ChEMBL + 3D | Streams censored (`=`, `>`, `<`) **hERG**, **Cav1.2**, **Nav1.5** bioactivities, zero-overlap Bemis–Murcko split, pH 7.4 protonation & MMFF94s Boltzmann 3D conformers |
| **2** | **WP2** FDA-CiPA 28 ODE Validation | GPU-vectorized O'Hara–Rudy cardiomyocyte ODE (112 trajectories), ROC-AUC vs hERG-only, Verapamil multi-channel rescue figure |
| **3** | **WP3** SE(3)-Equivariant PaiNN GPU | Trains 3D PaiNN teacher with censored Tobit NLL, bfloat16/fp16 AMP, Boltzmann conformer sampling & 90% Mondrian Conformal Prediction |
| **4** | **WP4** Knowledge Distillation | Compresses 3D PaiNN teacher into a sub-2 ms edge student (`CardioPhysStudentNet`) via closed-form Gaussian KL + Tobit NLL |
| **5** | **WP5** Closed-Loop Molecular Repair | Automatically detoxifies withdrawn drugs (**Terfenadine**, **Astemizole**, **Cisapride**) using the distilled WP4 → WP2 `P(TdP)` oracle |

> **Speed Mode:** Default `MAX_PER_CHANNEL = 3000` finishes the entire 5-WP pipeline in **~15 minutes** on Colab Pro L4/A100. Set `MAX_PER_CHANNEL = None` for the full ChEMBL v37 corpus.

In [ ]:
# @title 0. Master Configuration & Environment Setup (Edit Parameters Here)
import os, shutil, subprocess, sys, time
from pathlib import Path

# ---- User-Configurable Pipeline Controls ----
MAX_PER_CHANNEL = 3000       # @param {type:'raw'}  (3000 = ~15 min fast run; None = full ChEMBL v37)
NUM_CONFS = 5                # @param {type:'integer'}  (5 for fast mode; 10 for full paper run)
WP3_EPOCHS = 35              # @param {type:'integer'}
WP3_BATCH_SIZE = 64          # @param {type:'integer'}
WP4_EPOCHS = 45              # @param {type:'integer'}
ENABLE_DRIVE_BACKUP = True   # @param {type:'boolean'}  (optional & non-fatal if Drive is skipped)
REUSE_CACHED_WP1 = True      # @param {type:'boolean'}  (reuse WP1 data if already present locally/Drive)

PIPELINE_T0 = time.perf_counter()
REPO_DIR = Path('/content/Admet-Engine')

print('=' * 72)
print('  ADMET-Engine v3.0 (CardioPhys-3D) — 1-Click Master Pipeline')
print('=' * 72)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JakobFerschin/Admet-Engine.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'rdkit', 'torch-geometric', 'joblib', 'scikit-learn', 'scipy',
     'requests', 'matplotlib', 'pandas', 'tqdm'],
    check=True,
)

# Ensure canonical directories exist
DATA_DIR = REPO_DIR / 'data' / 'processed'
MODELS_DIR = REPO_DIR / 'models'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATA_CSV = DATA_DIR / 'cardiophys_multichannel_v3.csv'
CONF_PT = DATA_DIR / 'cardiophys_3d_conformers_v3.pt'
TEACHER_CKPT = MODELS_DIR / 'cardiophys_3d_v3.pt'
TEACHER_METRICS = MODELS_DIR / 'cardiophys_3d_v3_metrics.json'
STUDENT_CKPT = MODELS_DIR / 'cardiophys_student_v3.pt'
STUDENT_METRICS = MODELS_DIR / 'cardiophys_student_v3_metrics.json'
REPAIR_CSV = MODELS_DIR / 'wp5_repair_benchmark.csv'

src_path = str((REPO_DIR / 'src').resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[Setup] PyTorch {torch.__version__} | Device: {DEVICE} | CPUs: {os.cpu_count()}')
if torch.cuda.is_available():
    print(f'[Setup] GPU: {torch.cuda.get_device_name(0)} | bfloat16: {torch.cuda.is_bf16_supported()}')
else:
    print('[Setup] WARNING: Running on CPU. Switch Runtime -> GPU (T4/L4/A100) for fast WP3 training!')

# Optional, non-fatal Google Drive mount & cache helper
DRIVE_DIR = Path('/content/drive/MyDrive/CardioPhys3D_v3')
DRIVE_MOUNTED = False
if ENABLE_DRIVE_BACKUP:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        DRIVE_MOUNTED = True
        print(f'[Drive] Mounted successfully -> {DRIVE_DIR}')
    except Exception as e:
        print(f'[Drive] Mount skipped (non-fatal, continuing locally): {e}')

def backup_to_drive(*paths: Path) -> None:
    if not DRIVE_MOUNTED:
        return
    for p in paths:
        if p.exists():
            shutil.copy2(p, DRIVE_DIR / p.name)
            print(f'[Drive] Backed up {p.name} -> {DRIVE_DIR / p.name}')

# Optionally restore cached WP1 artifacts from Drive if requested
if REUSE_CACHED_WP1 and DRIVE_MOUNTED:
    for dest, candidates in [
        (DATA_CSV, ['cardiophys_multichannel_v3.csv', 'v3_multichannel_chembl37.csv']),
        (CONF_PT, ['cardiophys_3d_conformers_v3.pt', 'v3_conformers_3d.pt']),
    ]:
        if not dest.exists():
            for fname in candidates:
                src = DRIVE_DIR / fname
                if src.exists():
                    shutil.copy2(src, dest)
                    print(f'[Drive] Restored cached {fname} -> {dest}')
                    break

In [ ]:
# @title 1. Stage 1 (WP1) — ChEMBL v37 Multi-Channel Extraction & pH 7.4 3D Conformers
import time
import pandas as pd
import torch
from joblib import Parallel, delayed
from data.multichannel_extractor import extract_from_rest_api, build_multichannel_dataset
from data.conformer_3d import generate_3d_ensemble

t_wp1 = time.perf_counter()

if REUSE_CACHED_WP1 and DATA_CSV.exists():
    df_mc = pd.read_csv(DATA_CSV)
    print(f'[WP1] Reusing cached dataset: {DATA_CSV} ({len(df_mc):,} compounds)')
else:
    print(f'[WP1] Streaming hERG / Cav1.2 / Nav1.5 from ChEMBL REST API (MAX_PER_CHANNEL={MAX_PER_CHANNEL})...')
    raw_rows = extract_from_rest_api(max_per_target=MAX_PER_CHANNEL)
    df_mc = build_multichannel_dataset(raw_rows)
    assert not df_mc.empty, 'ChEMBL REST API extraction returned 0 valid rows!'
    df_mc.to_csv(DATA_CSV, index=False)
    print(f'[WP1] Saved {len(df_mc):,} unique compounds to {DATA_CSV}')
    backup_to_drive(DATA_CSV)

print('[WP1] Scaffold split counts:', df_mc['split'].value_counts().to_dict())

if REUSE_CACHED_WP1 and CONF_PT.exists():
    valid_3d = torch.load(str(CONF_PT), map_location='cpu', weights_only=False)
    print(f'[WP1] Reusing cached 3D conformers: {CONF_PT} ({len(valid_3d):,} molecules)')
else:
    smiles_list = df_mc['canonical_smiles'].tolist()
    print(f'[WP1] Generating pH 7.4 MMFF94s 3D conformers (NUM_CONFS={NUM_CONFS}) for {len(smiles_list):,} compounds...')

    def _featurize_one(smi):
        ens = generate_3d_ensemble(smi, num_confs=NUM_CONFS, protonate=True, num_threads=1)
        return None if ens is None else ens.to_torch_dict()

    valid_3d = {}
    chunk_size = 500
    t_c = time.perf_counter()
    for start in range(0, len(smiles_list), chunk_size):
        chunk = smiles_list[start:start + chunk_size]
        chunk_res = Parallel(n_jobs=-1, prefer='processes')(
            delayed(_featurize_one)(smi) for smi in chunk
        )
        for smi, r in zip(chunk, chunk_res):
            if r is not None:
                valid_3d[smi] = r
        done = min(start + chunk_size, len(smiles_list))
        print(f'  [WP1-3D] {done:,}/{len(smiles_list):,} processed ({len(valid_3d):,} valid) | {time.perf_counter() - t_c:.1f}s', flush=True)

    assert len(valid_3d) > 0, '3D conformer generation produced 0 valid molecules!'
    torch.save(valid_3d, CONF_PT)
    print(f'[WP1] Saved {len(valid_3d):,} 3D conformer ensembles to {CONF_PT}')
    backup_to_drive(CONF_PT)

print(f'[WP1] Completed in {time.perf_counter() - t_wp1:.1f}s')

In [ ]:
# @title 2. Stage 2 (WP2) — GPU-Vectorized FDA-CiPA 28 Cardiomyocyte ODE Validation & Rescue Plot
import time
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import roc_auc_score
from physiology import CardiomyocyteSimulator, ChannelBlockProfile, CIPA_28_DRUGS

t_wp2 = time.perf_counter()
sim = CardiomyocyteSimulator(dt_ms=0.2, n_beats=2).to(DEVICE)

def _t(vals):
    return torch.tensor([1e6 if v is None else float(v) for v in vals], dtype=torch.float32, device=DEVICE)

out = sim.simulate_cipa_protocol(
    cmax_free_uM=_t([d.cmax_free_uM for d in CIPA_28_DRUGS]),
    ic50_herg_uM=_t([d.ic50_herg_uM for d in CIPA_28_DRUGS]),
    ic50_cav12_uM=_t([d.ic50_cav12_uM for d in CIPA_28_DRUGS]),
    ic50_nav15_late_uM=_t([d.ic50_nav15_late_uM for d in CIPA_28_DRUGS]),
    ic50_nav15_peak_uM=_t([d.ic50_nav15_peak_uM for d in CIPA_28_DRUGS]),
    h_herg=_t([d.h_herg for d in CIPA_28_DRUGS]),
    h_cav12=_t([d.h_cav12 for d in CIPA_28_DRUGS]),
    h_nav15_late=_t([d.h_nav15_late for d in CIPA_28_DRUGS]),
    h_nav15_peak=_t([d.h_nav15_peak for d in CIPA_28_DRUGS]),
)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()

qnets = out['mean_qnet'].detach().cpu().numpy()
risks = [d.tdp_risk for d in CIPA_28_DRUGS]
y_hi_int = [1 if r in ('High', 'Intermediate') else 0 for r in risks]
y_hi_lo = [1 if r == 'High' else 0 for r in risks if r != 'Intermediate']
q_hi_lo = [-q for q, r in zip(qnets, risks) if r != 'Intermediate']
herg_ratio = [d.cmax_free_uM / (d.ic50_herg_uM or 1e6) for d in CIPA_28_DRUGS]
herg_hi_lo = [hr for hr, r in zip(herg_ratio, risks) if r != 'Intermediate']

cipa_auc_all = roc_auc_score(y_hi_int, -qnets)
cipa_auc_hilo = roc_auc_score(y_hi_lo, q_hi_lo)
print(f'[WP2] Simulated 112 FDA-CiPA ODE trajectories on {DEVICE} in {time.perf_counter() - t_wp2:.2f}s')
print(f'[WP2] CiPA-ODE (-qNet) ROC-AUC (High+Int vs Low): {cipa_auc_all:.4f} (hERG-only: {roc_auc_score(y_hi_int, herg_ratio):.4f})')
print(f'[WP2] CiPA-ODE (-qNet) ROC-AUC (High vs Low):     {cipa_auc_hilo:.4f} (hERG-only: {roc_auc_score(y_hi_lo, herg_hi_lo):.4f})')

# Mechanistic multi-channel rescue figure (Verapamil vs Dofetilide)
conc = torch.tensor([0.0, 0.008, 0.324, 0.324], device=DEVICE)
herg = torch.tensor([1e6, 0.0049, 0.499, 0.499], device=DEVICE)
cav12 = torch.tensor([1e6, 1e6, 1e6, 0.202], device=DEVICE)
res = sim.simulate(
    ChannelBlockProfile.from_ic50_uM(conc_uM=conc, ic50_herg_uM=herg, ic50_cav12_uM=cav12),
    return_traces=True, trace_stride=2,
)
t_ms = res.time_ms.cpu().numpy()
v_mv = res.v_trace.cpu().numpy()
labels = [
    f'Control (qNet={res.qnet[0]:.4f}, APD90={res.apd90_ms[0]:.0f} ms)',
    f'Dofetilide 4x Cmax (High TdP; qNet={res.qnet[1]:.4f}, APD90={res.apd90_ms[1]:.0f} ms)',
    f'Verapamil hERG-only (False Positive; qNet={res.qnet[2]:.4f}, APD90={res.apd90_ms[2]:.0f} ms)',
    f'Verapamil Full hERG+Cav1.2 (Rescued Safe; qNet={res.qnet[3]:.4f}, APD90={res.apd90_ms[3]:.0f} ms)',
]
plt.figure(figsize=(9, 4.3), dpi=140)
for i, (col, ls) in enumerate(zip(['#64748B', '#A51C30', '#B45309', '#0F766E'], ['--', '-', ':', '-'])):
    plt.plot(t_ms, v_mv[i], label=labels[i], color=col, ls=ls, lw=2.0)
plt.xlim(0, 800)
plt.xlabel('Time (ms)')
plt.ylabel('Membrane Potential $V_m$ (mV)')
plt.title('CardioPhys-3D Mechanistic Proof: Verapamil Multi-Channel Rescue (ORd-CiPA)')
plt.grid(alpha=0.3)
plt.legend(fontsize=8.5, frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# @title 3. Stage 3 (WP3) — SE(3)-Equivariant PaiNN 3D GPU Training (Tobit Censored NLL + Mondrian CP)
import json, subprocess, sys, time

t_wp3 = time.perf_counter()
print(f'[WP3] Launching SE(3)-Equivariant PaiNN training for {WP3_EPOCHS} epochs...')

subprocess.run([
    sys.executable, 'src/training/train_wp3_3d.py',
    '--csv', str(DATA_CSV),
    '--conformers', str(CONF_PT),
    '--epochs', str(WP3_EPOCHS),
    '--batch-size', str(WP3_BATCH_SIZE),
    '--hidden-dim', '128',
    '--num-layers', '3',
    '--lr', '5e-4',
    '--out-dir', str(MODELS_DIR),
], check=True)

wp3_report = json.loads(TEACHER_METRICS.read_text(encoding='utf-8'))
print(f'[WP3] Completed in {time.perf_counter() - t_wp3:.1f}s')
print(json.dumps(wp3_report['test_metrics'], indent=2))
backup_to_drive(TEACHER_CKPT, TEACHER_METRICS)

In [ ]:
# @title 4. Stage 4 (WP4) — Knowledge Distillation (3D PaiNN Teacher -> Sub-2ms Edge Student)
import json, subprocess, sys, time

t_wp4 = time.perf_counter()
print(f'[WP4] Distilling 3D PaiNN teacher -> edge student for {WP4_EPOCHS} epochs...')

subprocess.run([
    sys.executable, 'src/training/distill_wp4.py',
    '--data-csv', str(DATA_CSV),
    '--conformers-pt', str(CONF_PT),
    '--teacher-ckpt', str(TEACHER_CKPT),
    '--out-dir', str(MODELS_DIR),
    '--epochs', str(WP4_EPOCHS),
    '--batch-size', '128',
    '--distill-weight', '0.55',
], check=True)

wp4_report = json.loads(STUDENT_METRICS.read_text(encoding='utf-8'))
print(f'[WP4] Completed in {time.perf_counter() - t_wp4:.1f}s | Edge CPU latency: {wp4_report["single_mol_cpu_latency_ms"]:.3f} ms/mol')
print(json.dumps(wp4_report['test_metrics'], indent=2))
backup_to_drive(STUDENT_CKPT, STUDENT_METRICS)

In [ ]:
# @title 5. Stage 5 (WP5) — Closed-Loop Molecular Repair (Terfenadine, Astemizole, Cisapride)
import time
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import Draw
from models.distillation import CardioPhysEndToEndPipeline, CardioPhysStudentNet
from physiology.cipa28 import get_cipa28_drug
from repair.molecular_repair import ClosedLoopRepairAgent

t_wp5 = time.perf_counter()
ckpt = torch.load(str(STUDENT_CKPT), map_location='cpu', weights_only=False)
student = CardioPhysStudentNet(**ckpt['config'])
student.load_state_dict(ckpt['model_state_dict'])

pipe = CardioPhysEndToEndPipeline(student=student, device=str(DEVICE))
oracle_name = 'CardioPhys-3D Distilled Student -> ORd-CiPA P(TdP)'

def risk_oracle(smiles_list):
    preds = pipe.predict_smiles(smiles_list, conc_uM=1.0)
    return [p.tdp_risk_score if p is not None else 1.0 for p in preds]

agent = ClosedLoopRepairAgent(
    risk_oracle=risk_oracle,
    min_tanimoto=0.55,
    max_sa_score=4.8,
    min_qed=0.25,
    require_zero_pains=True,
)

targets = ['Terfenadine', 'Astemizole', 'Cisapride']
rows, grid_mols, grid_legends = [], [], []

for name in targets:
    drug = get_cipa28_drug(name)
    rep = agent.repair(drug.smiles, n_rounds=2, beam_width=5, top_k=3)
    seed = rep['seed']
    cands = rep.get('repaired_candidates', [])
    if not cands:
        print(f'[WP5] No candidate passed all MedChem constraints for {name}.')
        continue
    best = cands[0]
    rows.append({
        'Drug': name,
        'Oracle': oracle_name,
        'Seed Risk': seed['risk_score'],
        'Repaired Risk': best.risk_score,
        'Delta Risk': best.delta_risk,
        'Tanimoto': best.tanimoto,
        'Seed QED -> New QED': f"{seed['qed']} -> {best.qed}",
        'SA-Score': best.sa_score,
        'cLogP': f"{seed['clogp']} -> {best.clogp}",
        'TPSA': f"{seed['tpsa']} -> {best.tpsa}",
        'Edit Path': ' -> '.join(best.edit_path),
        'Repaired SMILES': best.smiles,
    })
    grid_mols.extend([Chem.MolFromSmiles(seed['smiles']), Chem.MolFromSmiles(best.smiles)])
    grid_legends.extend([
        f"{name} (Risk={seed['risk_score']:.3f})",
        f"{name} Repaired (Risk={best.risk_score:.3f}, d={best.delta_risk:+.3f})",
    ])

df_res = pd.DataFrame(rows)
df_res.to_csv(REPAIR_CSV, index=False)
backup_to_drive(REPAIR_CSV)

display(df_res)
if grid_mols:
    display(Draw.MolsToGridImage(grid_mols, molsPerRow=2, subImgSize=(420, 280), legends=grid_legends))

total_min = (time.perf_counter() - PIPELINE_T0) / 60.0
print('=' * 72)
print(f'  CardioPhys-3D ALL-IN-ONE PIPELINE COMPLETE in {total_min:.1f} min!')
print(f'  WP2 CiPA-28 High vs Low ROC-AUC: {cipa_auc_hilo:.4f}')
print(f'  WP4 Edge Student Latency:        {wp4_report["single_mol_cpu_latency_ms"]:.3f} ms/mol')
print(f'  Artifacts saved to:              {MODELS_DIR}')
if DRIVE_MOUNTED:
    print(f'  Drive backup synced to:          {DRIVE_DIR}')
print('=' * 72)